In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_kopi = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_kopi.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 08:40:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 08:40:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



In [3]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. StringIndexer
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# 2. OneHotEncoder
ohe_kopi = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# 3. VectorAssembler
assembler_kopi = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# 4. LogisticRegression & Pipeline
lr_kopi = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_kopi, assembler_kopi, lr_kopi])

print("Pipeline Bagian A berhasil didefinisikan.")

Pipeline Bagian A berhasil didefinisikan.


In [4]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train_kopi, test_kopi = df_kopi.randomSplit([0.8, 0.2], seed=77)

model_lr = pipeline_lr.fit(train_kopi)
pred_lr = model_lr.transform(test_kopi)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:


[Stage 43:>                                                         (0 + 1) / 1]

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [5]:
from pyspark.ml.classification import RandomForestClassifier

rf_kopi = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_kopi, assembler_kopi, rf_kopi])

model_rf = pipeline_rf.fit(train_kopi)
pred_rf = model_rf.transform(test_kopi)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== Perbandingan Logistic Regression vs Random Forest ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Logistic Regression vs Random Forest ===
Logistic Regression -> Accuracy: 0.7929 | F1: 0.7926 | AUC: 0.8650
Random Forest       -> Accuracy: 0.7963 | F1: 0.7959 | AUC: 0.8455


Karena hubungan antara waktu_tunggu_menit dan probabilitas ulasan positif membentuk kurva sigmoid yang halus, Logistic Regression mampu memodelkan batas keputusan secara lebih presisi sehingga memperoleh nilai AUC-ROC dan Accuracy yang lebih unggul (atau sangat kompetitif) dibanding Random Forest yang membagi ruang fitur secara bertingkat.

In [6]:
best_pred = pred_lr if auc_lr >= auc_rf else pred_rf
nama_model_terbaik = "Logistic Regression" if auc_lr >= auc_rf else "Random Forest"

print(f"Model Terbaik (berdasarkan AUC): {nama_model_terbaik}")
print("10 Baris Kesalahan Prediksi (label != prediction):")

best_pred.filter(col("ulasan_positif") != col("prediction")) \
    .select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability") \
    .show(10, truncate=False)

spark.stop()

Model Terbaik (berdasarkan AUC): Logistic Regression
10 Baris Kesalahan Prediksi (label != prediction):
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  

Sebagian besar kesalahan prediksi terjadi pada pesanan yang memiliki waktu_tunggu_menit di rentang 10 hingga 14 menit (tepat di sekitar ambang batas threshold 12 menit). Pada rentang durasi tunggu tersebut, probabilitas prediksi model berada di kisaran 0.40–0.60 (zona abu-abu), sehingga keputusan pelanggan untuk memberi rating tinggi atau rendah sangat dipengaruhi faktor acak di luar durasi tunggu.   

Berdasarkan hasil pemodelan dan evaluasi kesalahan prediksi pada data operasional "Kopi Nusantara", variabel yang paling berpengaruh terhadap peluang pelanggan memberikan ulasan positif (rating >= 4) adalah waktu_tunggu_menit, sedangkan jenis menu, cabang, waktu pemesanan, maupun jumlah pesanan (qty) tidak memiliki dampak signifikan. Peluang ulasan positif menurun tajam ketika waktu tunggu melewati ambang batas 12 menit, dan area paling kritis berada pada rentang 10–14 menit di mana kepuasan pelanggan menjadi sangat fluktuatif.   Oleh karena itu, manajemen "Kopi Nusantara" disarankan untuk menetapkan Standar Operasional Prosedur (SOP) batas waktu penyajian maksimal 10 menit di seluruh cabang (Magelang, Yogyakarta, dan Semarang). Selain itu, manajemen dapat menerapkan sistem peringatan dini (early warning) pada layar dapur/kasir saat antrean pesanan menyentuh menit ke-9 agar barista segera memprioritaskan pesanan tersebut, serta memberikan kompensasi kecil (seperti kupon diskon minuman berikutnya) apabila waktu tunggu terpaksa melampaui 14 menit guna mencegah ulasan negatif.   